## Walk-Forward Results — What the Columns Mean

Each row = **one walk-forward window** for one `max_num_components`.

### 1. Periods

- `calibration_start`, `calibration_end`  
  Historical data used for optimization.

- `test_start`, `test_end`  
  The following **out-of-sample (OOS)** period. It is never used for threshold selection.

---

### 2. Stage 1 — Broad Monte Carlo

Stage 1 performs `N_SIMULATIONS_STAGE1` random threshold combinations over the **complete threshold space**.

- `stage1_robust_buy_thr`
- `stage1_robust_sell_thr`

  → Robust thresholds calculated from the **top 10% Stage-1 simulations**.

- `stage1_top10_buy_std`
- `stage1_top10_sell_std`

  → Standard deviation of the buy/sell thresholds among those top 10%.

These describe the **promising region** found by Stage 1.

---

### 3. Stage 2 — Focused Monte Carlo

Stage 2 uses the Stage-1 robust thresholds as its **center** and searches locally around them.

- `stage2_buy_min`, `stage2_buy_max`
- `stage2_sell_min`, `stage2_sell_max`

  → Actual threshold range searched by Stage 2.

- `stage2_n`

  → Number of Stage-2 simulations performed.

---

### 4. Four Optimization Methods

Each Stage-2 simulation is evaluated using four criteria:

#### Sharpe
Selects the threshold combination with the highest **Sharpe ratio**.

- `sharpe_buy_thr`
- `sharpe_sell_thr`
- `sharpe_calibration`
- `sharpe_calibration_return`

#### Calmar
Selects the combination with the highest **Calmar ratio**.

- `calmar_buy_thr`
- `calmar_sell_thr`
- `calmar_calibration`
- `calmar_calibration_return`

#### Return / Risk
Selects the combination with the highest **return relative to risk**.

- `return_risk_buy_thr`
- `return_risk_sell_thr`
- `return_risk_calibration`
- `return_risk_calibration_return`

#### Robust
Does **not** select simply the single best simulation.

It looks at the **top-performing region** and derives robust thresholds from that region.

- `robust_buy_thr`
- `robust_sell_thr`
- `robust_calibration_sharpe`
- `robust_calibration_return`
- `robust_calibration_calmar`
- `robust_calibration_return_risk`

---

### 5. OOS Results

The four selected threshold pairs are then applied to the **next test period**, which was never used during optimization.

- `sharpe_oos_return`
- `calmar_oos_return`
- `return_risk_oos_return`
- `robust_oos_return`

These are the actual **out-of-sample returns**.

This is the most important part for judging whether the optimization generalizes.

---

### 6. Excess Return vs INDEX

`index_return`

→ Return of the DOW/INDEX during the same OOS period.

For each method:

- `sharpe_excess_return`
- `calmar_excess_return`
- `return_risk_excess_return`
- `robust_excess_return`

calculated as:

`strategy OOS return − index return`

Positive = strategy beat the index.

---

### 7. Stage-2 Distribution

- `stage2_buy_std`
- `stage2_sell_std`

→ Dispersion of the Stage-2 sampled thresholds.

- `stage2_actual_buy_min`
- `stage2_actual_buy_max`
- `stage2_actual_sell_min`
- `stage2_actual_sell_max`

→ Actual minimum/maximum thresholds generated in Stage 2.

---

### In short

The complete process is:

**Stage 1 → find promising region → Stage 2 → optimize 4 criteria → select thresholds → test on unseen OOS data**

The key comparison is therefore:

**`sharpe_oos_return` vs `calmar_oos_return` vs `return_risk_oos_return` vs `robust_oos_return` vs `index_return`.**

And especially whether the **robust OOS performance** remains good across many different walk-forward windows.

In [ ]:
Historical data
      │
      ▼
Walk-forward calibration window
      │
      ▼
Stage 1 Monte-Carlo
      │
      │ Broad search
      ▼
Top-performing threshold region
      │
      ▼
Robust center + threshold dispersion
      │
      ▼
Stage 2 Monte-Carlo
      │
      │ Focused search
      ▼
 ┌──────────────┬──────────────┬──────────────┬──────────────┐
 │    Sharpe    │    Calmar    │ Return/Risk  │    Robust    │
 └──────────────┴──────────────┴──────────────┴──────────────┘
      │
      ▼
Four threshold pairs
      │
      ▼
Following OOS period
      │
      ├── Sharpe OOS
      ├── Calmar OOS
      ├── Return/Risk OOS
      └── Robust OOS
      │
      ▼
Compare against INDEX
      │
      ▼
Move calibration window forward
      │
      ▼
Repeat